## 1 import Libararies

In [1]:
import numpy as np 
import pandas as pd

##  2 Load Dataset

In [2]:
df = pd.read_csv("nykaa_campaign_data.csv")

In [3]:
df.head()

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date
0,NY-CMP-1000,Social Media,College Students,21,"WhatsApp, YouTube",57804,6156,3616,2355,1867515,111.03,6.14,Hindi,20.98,College Students,29-04-2025
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18,YouTube,91801,3321,1971,1357,1046247,180.83,3.26,Hindi,7.24,College Students,06-04-2025
2,NY-CMP-1002,Influencer,Youth,23,"WhatsApp, Google, YouTube",15536,2182,952,755,197055,90.60,1.88,English,25.03,College Students,14-01-2025
3,NY-CMP-1003,Email,Working Women,18,"YouTube, Facebook, Instagram",88114,8413,2231,947,376906,249.07,0.60,Hindi,13.15,College Students,04-06-2025
4,NY-CMP-1004,Paid Ads,College Students,10,"Facebook, Instagram",96871,3743,2060,1258,518296,228.60,0.80,Hindi,7.29,Tier 2 City Customers,29-12-2024


In [4]:
df.shape

(55555, 16)

In [5]:
df.dtypes

Campaign_ID          object
Campaign_Type        object
Target_Audience      object
Duration              int64
Channel_Used         object
Impressions           int64
Clicks                int64
Leads                 int64
Conversions           int64
Revenue               int64
Acquisition_Cost    float64
ROI                 float64
Language             object
Engagement_Score    float64
Customer_Segment     object
Date                 object
dtype: object

In [6]:
print("Missing values:\n", df.isna().sum())          
print("Duplicate rows:", df.duplicated().sum())       
print("Duplicate IDs:", df["Campaign_ID"].duplicated().sum())

Missing values:
 Campaign_ID         0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Impressions         0
Clicks              0
Leads               0
Conversions         0
Revenue             0
Acquisition_Cost    0
ROI                 0
Language            0
Engagement_Score    0
Customer_Segment    0
Date                0
dtype: int64
Duplicate rows: 0
Duplicate IDs: 0


## 3 Clean Data

In [7]:
df.columns = df.columns.str.strip()      
text_cols = df.select_dtypes(include=["object", "string"]).columns   
for col in text_cols:
    df[col] = df[col].str.strip()        

df = df.drop_duplicates()                

## 4 Fix Date

In [8]:
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55555 entries, 0 to 55554
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Campaign_ID       55555 non-null  object        
 1   Campaign_Type     55555 non-null  object        
 2   Target_Audience   55555 non-null  object        
 3   Duration          55555 non-null  int64         
 4   Channel_Used      55555 non-null  object        
 5   Impressions       55555 non-null  int64         
 6   Clicks            55555 non-null  int64         
 7   Leads             55555 non-null  int64         
 8   Conversions       55555 non-null  int64         
 9   Revenue           55555 non-null  int64         
 10  Acquisition_Cost  55555 non-null  float64       
 11  ROI               55555 non-null  float64       
 12  Language          55555 non-null  object        
 13  Engagement_Score  55555 non-null  float64       
 14  Customer_Segment  5555

## 5 Cheking Negetive Values & Wrong rows

In [10]:
bad = df[(df["Clicks"] > df["Impressions"]) |
         (df["Leads"] > df["Clicks"]) |
         (df["Conversions"] > df["Leads"])]
print("Wrong rows:", len(bad))       # ithe 0 yeil, mhanje data barobar aahe
df = df.drop(bad.index)                  # chukichya rows asel tar kadhun taka

num_cols = ["Impressions", "Clicks", "Leads", "Conversions", "Revenue", "Acquisition_Cost"]
print("Negative values:", (df[num_cols] < 0).sum().sum())   # minus value nahi na?

Wrong rows: 0
Negative values: 0


## 6 Feature Engineering

In [11]:
df["Total_Cost"] = (df["Acquisition_Cost"] * df["Conversions"]).round(2)
df["Profit"]     = (df["Revenue"] - df["Total_Cost"]).round(2)

In [12]:
df["CTR_pct"]             = (df["Clicks"] / df["Impressions"] * 100).round(2)   # click kiti %
df["Click_to_Lead_pct"]   = (df["Leads"] / df["Clicks"] * 100).round(2)
df["Lead_to_Conv_pct"]    = (df["Conversions"] / df["Leads"] * 100).round(2)
df["Conversion_Rate_pct"] = (df["Conversions"] / df["Clicks"] * 100).round(2)

In [13]:
# Date madhun parts kadhne (charts sathi upyogi)
df["Year"]       = df["Date"].dt.year
df["Month_Num"]  = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")        # Jan, Feb ...
df["Year_Month"] = df["Date"].dt.strftime("%Y-%m")     # 2025-04
df["Quarter"]    = "Q" + df["Date"].dt.quarter.astype(str)

In [14]:
# Kiti channels vaparle (comma var split karun mojle)
df["Num_Channels"] = df["Channel_Used"].str.split(",").str.len()

In [15]:
# ROI cha group: Loss / Low / Medium / High
df["ROI_Category"] = pd.cut(
    df["ROI"],
    bins=[-np.inf, 0, 1, 3, np.inf],
    labels=["Loss", "Low (0-1)", "Medium (1-3)", "High (3+)"])

In [17]:
# Target_Audience ani Customer_Segment same aahe ka?
df["Audience_Match"] = np.where(df["Target_Audience"] == df["Customer_Segment"],
                                "Match", "Mismatch")

## 7 Cheking outlier

In [18]:
def iqr_flag(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

df["Outlier_CAC"] = iqr_flag(df["Acquisition_Cost"])
df["Outlier_ROI"] = iqr_flag(df["ROI"])
print("CAC outliers:", df["Outlier_CAC"].sum(), "| ROI outliers:", df["Outlier_ROI"].sum())

CAC outliers: 4845 | ROI outliers: 4081


## 8 Table Channel

In [19]:
channel = df[["Campaign_ID", "Channel_Used"]].copy()
channel["Channel"] = channel["Channel_Used"].str.split(",")   # list banvli
channel = channel.explode("Channel")                          # list chya pratyek item la navin row
channel["Channel"] = channel["Channel"].str.strip()
channel = channel[["Campaign_ID", "Channel"]]

In [20]:
df

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,...,Year,Month_Num,Month_Name,Year_Month,Quarter,Num_Channels,ROI_Category,Audience_Match,Outlier_CAC,Outlier_ROI
0,NY-CMP-1000,Social Media,College Students,21,"WhatsApp, YouTube",57804,6156,3616,2355,1867515,...,2025,4,Apr,2025-04,Q2,2,High (3+),Match,False,False
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18,YouTube,91801,3321,1971,1357,1046247,...,2025,4,Apr,2025-04,Q2,1,High (3+),Mismatch,False,False
2,NY-CMP-1002,Influencer,Youth,23,"WhatsApp, Google, YouTube",15536,2182,952,755,197055,...,2025,1,Jan,2025-01,Q1,3,Medium (1-3),Mismatch,False,False
3,NY-CMP-1003,Email,Working Women,18,"YouTube, Facebook, Instagram",88114,8413,2231,947,376906,...,2025,6,Jun,2025-06,Q2,3,Low (0-1),Mismatch,False,False
4,NY-CMP-1004,Paid Ads,College Students,10,"Facebook, Instagram",96871,3743,2060,1258,518296,...,2024,12,Dec,2024-12,Q4,2,Low (0-1),Mismatch,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55550,NY-CMP-56550,SEO,Youth,16,"Instagram, Email",31041,3849,1012,639,162945,...,2025,2,Feb,2025-02,Q1,2,Loss,Mismatch,False,False
55551,NY-CMP-56551,Paid Ads,Tier 2 City Customers,26,Facebook,27335,3715,2184,1006,792728,...,2024,11,Nov,2024-11,Q4,1,High (3+),Mismatch,False,True
55552,NY-CMP-56552,Influencer,Tier 2 City Customers,23,"WhatsApp, Facebook, Instagram",43481,6447,3751,1668,383640,...,2024,10,Oct,2024-10,Q4,3,High (3+),Mismatch,False,False
55553,NY-CMP-56553,Social Media,College Students,16,"Facebook, Email",33530,2518,584,299,194051,...,2025,5,May,2025-05,Q2,2,Medium (1-3),Mismatch,False,False


## 9 Save Export

In [21]:
df.to_csv("nykaa_clean.csv", index=False)         # main clean file
channel.to_csv("nykaa_channel.csv", index=False)  # channel file
print("Done! Clean rows:", len(df), "| Channel rows:", len(channel))

Done! Clean rows: 55555 | Channel rows: 110903
